## Cell 0 — Colab setup  ← run this first if using Colab, skip locally

Mounts Google Drive, extracts the zip, and installs dependencies.

**Before running:** upload `pollinator_colab.zip` to the root of your Google Drive.

**To create the zip locally:**
```bash
cd ~/Downloads/bachelor\ thesis/automated-ecological-image-analysis/ml-pipelines/notebooks
zip -r pollinator_colab.zip pollinator-classification/
```

The zip should contain:
```
pollinator-classification/
  Insects_images/
    e2e_evaluation_images/   ← your images (~1GB)
    e2e_yolo_annotations/    ← GT annotations
  models/
    binary_best.pth
    4group_insectnet.pth
    5group_efficientnet.pth
    5group_insectnet.pth
```

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 0 — COLAB SETUP  ← run this cell ONLY on Colab
# ════════════════════════════════════════════════════════════
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import zipfile, os, shutil
    from pathlib import Path

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    ZIP_PATH   = DRIVE_ROOT / 'pollinator-colab.zip'
    # Extract to local /content/ (faster, no Drive connection issues)
    EXTRACT_TO = Path('/content/pollinator-colab')

    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} to /content/ ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('✓ Extracted to /content/pollinator-colab')
    else:
        print(f'✓ Already extracted: {EXTRACT_TO}')

    # Install dependencies
    print('Installing dependencies...')
    os.system('pip install -q opencv-python-headless torch torchvision')
    print('✓ Dependencies ready')

    # Verify key paths
    BASE_DIR   = EXTRACT_TO
    IMAGE_ROOT = BASE_DIR / 'Insects_images' / 'e2e_evaluation_images'
    MODEL_DIR  = BASE_DIR / 'models'
    print(f'\nPaths:')
    print(f'  IMAGE_ROOT : {IMAGE_ROOT}  exists={IMAGE_ROOT.exists()}')
    print(f'  MODEL_DIR  : {MODEL_DIR}   exists={MODEL_DIR.exists()}')
    for m in ['binary_best.pth','4group_insectnet.pth',
              '5group_efficientnet.pth','5group_insectnet.pth']:
        p = MODEL_DIR / m
        print(f'  {m}: {"✓" if p.exists() else "✗ NOT FOUND"}')
else:
    print('Running locally — skip this cell.')


# Crop-Based Inference

Detects insects in all camera folders and classifies each candidate crop
through multiple pipelines simultaneously.

**When to use this notebook:**
- `MODE = 'preprocess'` — you have no models yet, just want to extract candidate crops for labeling
- `MODE = 'infer'` — you have trained models and want to run full detection + classification

**What you get:**
```
crop_results/{RUN_NAME}/{camera}/
  results.csv   ← one row per candidate crop
                   columns: bbox coords + one set of prediction columns per pipeline
                   e.g. two_stage__binary_label, five_class_eff__pollinator_type, ...
  crops/        ← all crop images (both insect and background predictions)
  debug/        ← annotated frames showing detected bboxes
crop_results/{RUN_NAME}/
  run_config.json  ← exact preprocessing params + pipelines used (for traceability)
```

**To compare different settings:** change `RUN_NAME` and edit `PREPROCESS_CONFIG` or `PIPELINES`.
Each run is stored separately so results never overwrite each other.

**Next step:** run `evaluate.ipynb` to compare results against ground truth.

## Cell 1 — Environment

Set your local path. **Only edit `BASE_DIR`** — everything else is derived from it.
On Colab, `BASE_DIR` is set automatically after mounting Drive.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT  ← only edit this cell
# ════════════════════════════════════════════════════════════
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/pollinator-colab')
else:
    BASE_DIR = Path('/Users/lianshi/Downloads/bachelor thesis'
                    '/automated-ecological-image-analysis'
                    '/ml-pipelines/notebooks/pollinator-classification')

IMAGE_ROOT        = BASE_DIR / 'Insects_images' / 'e2e_evaluation_images'
GT_ANN_ROOT       = BASE_DIR / 'Insects_images' / 'e2e_yolo_annotations'
MODEL_DIR         = BASE_DIR / 'models'
CROP_RESULTS_ROOT = BASE_DIR / 'Insects_images' / 'crop_results'
YOLO_RESULTS_ROOT = BASE_DIR / 'Insects_images' / 'yolo_results'
INSECTNET_W       = BASE_DIR / 'InsectNet' / 'model.pth'
LABELED_DIR       = BASE_DIR / 'Insects_images' / 'annotated_crops'

CROP_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
YOLO_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Env              : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR         : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'IMAGE_ROOT       : {IMAGE_ROOT}  exists={IMAGE_ROOT.exists()}')
print(f'MODEL_DIR        : {MODEL_DIR}  exists={MODEL_DIR.exists()}')


## Cell 2 — Imports + pipeline code

Run all cells in this section before editing the run config.
Contains all functions needed by the pipeline.
**Do not edit.**

In [ ]:
import sys, csv, re, time, json, datetime
from pathlib import Path
from collections import defaultdict
import cv2, numpy as np
from PIL import Image as PILImage
from PIL.ExifTags import TAGS


In [ ]:
DEFAULT_PREPROCESS_CONFIG = {
    # ── Background ──────────────────────────────────────────────
    'background_sample_size':    0,
    'rolling_window':            1,
    # ── Background subtraction ──────────────────────────────────
    'darker_threshold':          30,
    'diff_blur_kernel_size':     7,
    # ── Vegetation mask (HSV) ────────────────────────────────────
    'veg_hue_lo':25, 'veg_hue_hi':95,
    'veg_sat_lo':40, 'veg_val_lo':40,
    # ── Contour filtering ────────────────────────────────────────
    'min_contour_area':          400,
    'max_contour_area':          35000,
    'max_large_motion_area':     600000,
    'max_aspect_ratio':          5,
    'merge_dist':                20,
    'min_crop_px':               10,
    # ── Morphological kernels ────────────────────────────────────
    'kernel_open_size':          3,
    'kernel_close_size':         11,
    # ── Large motion ─────────────────────────────────────────────
    'enable_large_motion':       True,
    'large_motion_tile_sizes':   [320,512],
    'large_motion_tile_stride_frac':    0.65,
    'large_motion_max_tiles_per_size':  6,
    'large_motion_max_tiles_total':     10,
    'large_motion_tile_nms_iou':        0.35,
    'large_motion_min_fg_frac':         0.008,
    'large_motion_context_pad':         10,
    'large_motion_fallback_sizes':      [640],
    'large_motion_fallback_centers':    [(0.35,0.78),(0.50,0.78),(0.65,0.78)],
    'large_motion_max_fallbacks':       3,
    'large_motion_fallback_min_fg_frac':0.003,
    # ── Tile scoring ─────────────────────────────────────────────
    'tile_texture_norm':         80.0,
    'tile_fg_norm':              0.12,
    'tile_fg_penalty_thresh':    0.45,
    'tile_fg_penalty_factor':    0.75,
    'tile_score_fg_weight':      0.65,
    'tile_score_texture_weight': 0.35,
    # ── Crop sizing ───────────────────────────────────────────────
    'crop_small_eff_thresh':50,   'crop_medium_eff_thresh':120,
    'crop_small_multiplier':2.5,  'crop_medium_multiplier':1.8,  'crop_large_multiplier':1.2,
    'crop_small_min_px':50,       'crop_small_max_px':180,
    'crop_medium_min_px':80,      'crop_medium_max_px':260,
    'crop_large_min_px':140,      'crop_large_max_px':320,
    'crop_pad_ratio':0.15,
    # ── ROI ──────────────────────────────────────────────────────
    'use_roi':False, 'manual_roi':False,
    'marker_hue':(45,75), 'marker_sat_min':200, 'marker_val_min':100,
    'marker_min_area':200, 'marker_zone_radius':800,
    'near_flower_iou_threshold':0.1,
    # ── Strip ────────────────────────────────────────────────────
    'strip_height':120, 'strip_ocr_temperature':False,
    'strip_ocr_target_height':120, 'strip_temp_min':-50, 'strip_temp_max':60,
    # ── Quality ──────────────────────────────────────────────────
    'skip_flash':True, 'skip_foggy':True,
    'foggy_threshold':50, 'sunny_shutter_threshold':150,
    # ── Misc ─────────────────────────────────────────────────────
    'use_exif_sort':False, 'progress_every':50,
    'near_flower_iou_threshold':0.1,
    # ── Debug ────────────────────────────────────────────────────
    'debug_outputs':'annotate', 'debug_save_empty_frames':False,
    'debug_max_width':1024, 'debug_jpeg_quality':60,
}
print('✓ Default preprocessing config loaded.')


In [ ]:
import csv, re, time, json, datetime
from pathlib import Path
from collections import defaultdict
import cv2, numpy as np
from PIL import Image as PILImage
from PIL.ExifTags import TAGS

_EXIF_DT_TAG = next((k for k,v in TAGS.items() if v=='DateTimeOriginal'), None)
_exif_cache  = {}

def _exif_dt(path):
    key=str(path)
    if key in _exif_cache: return _exif_cache[key]
    try:
        img=PILImage.open(path)
        exif=img._getexif() if hasattr(img,'_getexif') else dict(img.getexif())
        val=exif.get(_EXIF_DT_TAG) if exif and _EXIF_DT_TAG else None
    except Exception: val=None
    _exif_cache[key]=val; return val

def filename_sort_key(p):
    p=Path(p); m=re.search(r'(\d+)',p.stem)
    return (0,int(m.group(1)),p.name) if m else (1,p.name)

def robust_sort_key(p):
    p=Path(p); dt=_exif_dt(p)
    if dt:
        try:
            d,t=dt.split(' ')
            return (0,tuple(int(x) for x in d.split(':')+t.split(':')),p.name)
        except Exception: pass
    return filename_sort_key(p)

def setup_roi(paths, cfg):
    first=cv2.imread(str(paths[0]))
    full=np.ones(first.shape[:2],dtype=np.uint8)*255
    if not cfg.get('use_roi',False):
        print('  ROI: disabled — full image')
        return full, None
    print('  ROI: enabled')
    return full, None  # marker logic omitted for brevity

def build_background(paths, cfg):
    n=cfg.get('background_sample_size',0)
    if not n: print('  Background: rolling window only'); return None
    step=max(1,len(paths)//n)
    frames=[f for f in (cv2.imread(str(p)) for p in paths[::step][:n]) if f is not None]
    print(f'  Background: {len(frames)} frames sampled')
    return np.median(frames,axis=0).astype(np.uint8) if frames else None

def _bbox_iou(a,b):
    ax,ay,aw,ah=a; bx,by,bw,bh=b
    iw=max(0,min(ax+aw,bx+bw)-max(ax,bx)); ih=max(0,min(ay+ah,by+bh)-max(ay,by))
    inter=iw*ih; union=aw*ah+bw*bh-inter
    return inter/union if union else 0.0

def _tile_score(image,mask,bbox,cfg):
    x,y,w,h=bbox; tm=mask[y:y+h,x:x+w]; ti=image[y:y+h,x:x+w]
    if tm.size==0 or ti.size==0: return 0.0
    fg=np.count_nonzero(tm)/float(w*h)
    tex=min(cv2.cvtColor(ti,cv2.COLOR_BGR2GRAY).std()/cfg.get('tile_texture_norm',80.0),1.0)
    fgs=min(fg/cfg.get('tile_fg_norm',0.12),1.0)
    if fg>cfg.get('tile_fg_penalty_thresh',0.45): fgs*=cfg.get('tile_fg_penalty_factor',0.75)
    return cfg.get('tile_score_fg_weight',0.65)*fgs+cfg.get('tile_score_texture_weight',0.35)*tex

def _nms_tiles(scored,max_n,iou_thr):
    sel=[]
    for sc,bb in sorted(scored,key=lambda x:x[0],reverse=True):
        if all(_bbox_iou(bb,b)<=iou_thr for _,b in sel): sel.append((sc,bb))
        if len(sel)>=max_n: break
    return sel

def _tile_large(image,mask,bbox,cfg):
    x,y,w,h=bbox; H,W=mask.shape[:2]; all_sc=[]; seen=set()
    for ts_ in [int(t) for t in cfg.get('large_motion_tile_sizes',[320,512])]:
        ts_=min(ts_,W,H)
        if ts_<=0: continue
        st=max(1,int(ts_*cfg.get('large_motion_tile_stride_frac',0.65)))
        sc_sz=[]
        for yy in list(range(y,max(y+h-ts_+1,y+1),st))+[y+h-ts_]:
            for xx in list(range(x,max(x+w-ts_+1,x+1),st))+[x+w-ts_]:
                xx=max(0,min(int(xx),W-ts_)) if W>ts_ else 0
                yy=max(0,min(int(yy),H-ts_)) if H>ts_ else 0
                tw_,th_=min(ts_,W-xx),min(ts_,H-yy)
                key=(xx,yy,tw_,th_)
                if key in seen or tw_<=0 or th_<=0: continue
                fg=np.count_nonzero(mask[yy:yy+th_,xx:xx+tw_])/float(tw_*th_)
                if fg<cfg.get('large_motion_min_fg_frac',0.008): continue
                seen.add(key); sc_sz.append((_tile_score(image,mask,key,cfg),key))
        all_sc.extend(_nms_tiles(sc_sz,cfg.get('large_motion_max_tiles_per_size',6),
                                  cfg.get('large_motion_tile_nms_iou',0.35)))
    return [b for _,b in _nms_tiles(all_sc,cfg.get('large_motion_max_tiles_total',10),
                                     cfg.get('large_motion_tile_nms_iou',0.35))]

def merge_nearby_bboxes(bboxes,dist=20,max_area=35000):
    if not bboxes: return []
    boxes=list(bboxes); changed=True
    while changed:
        changed=False; merged=[]; used=set()
        for i,(x1,y1,w1,h1) in enumerate(boxes):
            if i in used: continue
            gx1,gy1,gx2,gy2=x1,y1,x1+w1,y1+h1
            for j,(x2,y2,w2,h2) in enumerate(boxes):
                if j<=i or j in used: continue
                if max(0,max(x2,gx1)-min(x2+w2,gx2))<=dist and                    max(0,max(y2,gy1)-min(y2+h2,gy2))<=dist:
                    nx1,ny1=min(gx1,x2),min(gy1,y2)
                    nx2,ny2=max(gx2,x2+w2),max(gy2,y2+h2)
                    if (nx2-nx1)*(ny2-ny1)<=max_area:
                        gx1,gy1,gx2,gy2=nx1,ny1,nx2,ny2; used.add(j); changed=True
            used.add(i); merged.append((gx1,gy1,gx2-gx1,gy2-gy1))
        boxes=merged
    return boxes

def detect_visitor(image,bg,zone,cfg):
    gi=cv2.bitwise_and(cv2.cvtColor(image,cv2.COLOR_BGR2GRAY),
                       cv2.cvtColor(image,cv2.COLOR_BGR2GRAY),mask=zone)
    gb=cv2.bitwise_and(cv2.cvtColor(bg,cv2.COLOR_BGR2GRAY),
                       cv2.cvtColor(bg,cv2.COLOR_BGR2GRAY),mask=zone)
    diff=cv2.absdiff(gb,gi)
    k=int(cfg.get('diff_blur_kernel_size',7)); k=k if k%2 else k+1
    diff=cv2.GaussianBlur(diff,(k,k),0)
    _,mask=cv2.threshold(diff,cfg['darker_threshold'],255,cv2.THRESH_BINARY)
    mask=cv2.bitwise_and(mask,zone)
    hsv=cv2.cvtColor(image,cv2.COLOR_BGR2HSV)
    green=cv2.inRange(hsv,
        np.array([cfg.get('veg_hue_lo',25),cfg.get('veg_sat_lo',40),cfg.get('veg_val_lo',40)]),
        np.array([cfg.get('veg_hue_hi',95),255,255]))
    mask=cv2.bitwise_and(mask,cv2.bitwise_not(green))
    ko=cfg.get('kernel_open_size',3); kc=cfg.get('kernel_close_size',11)
    mask=cv2.morphologyEx(mask,cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(ko,ko)))
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(kc,kc)))
    cnts,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    normal=[]; dets=[]; max_n=cfg['max_contour_area']; max_l=cfg['max_large_motion_area']
    enable_lm=cfg.get('enable_large_motion',True)
    for c in cnts:
        area=cv2.contourArea(c)
        if area<cfg['min_contour_area']: continue
        x,y,w,h=cv2.boundingRect(c)
        if area>max_n:
            if area<=max_l and enable_lm:
                sb=(x,y,w,h)
                dets.append({'bbox':sb,'candidate_type':'large_motion_context','source_area':area,'static_suspect':False})
                for tb in _tile_large(image,mask,sb,cfg):
                    dets.append({'bbox':tb,'candidate_type':'large_motion_tile','source_area':area,'static_suspect':False})
            continue
        if max(w,h)/max(min(w,h),1)>cfg['max_aspect_ratio']: continue
        normal.append((area,(x,y,w,h)))
    for bb in merge_nearby_bboxes([b for _,b in sorted(normal,reverse=True)],
                                   cfg.get('merge_dist',20),max_n):
        x,y,w,h=bb
        if max(w,h)>=cfg.get('min_crop_px',10):
            dets.append({'bbox':bb,'candidate_type':'normal','source_area':w*h,'static_suspect':False})
    return dets

def crop_with_padding(image,det,cfg):
    x,y,w,h=det['bbox']; Hi,Wi=image.shape[:2]
    ct=det.get('candidate_type','normal')
    if ct=='large_motion_context':
        pad=cfg.get('large_motion_context_pad',10)
        return image[max(0,y-pad):min(Hi,y+h+pad),max(0,x-pad):min(Wi,x+w+pad)]
    if 'large_motion' in ct: return image[y:y+h,x:x+w]
    eff=(w*h)**0.5
    if eff<cfg.get('crop_small_eff_thresh',50):
        win=int(eff*cfg.get('crop_small_multiplier',2.5))
        win=max(cfg.get('crop_small_min_px',50),min(win,cfg.get('crop_small_max_px',180)))
    elif eff<cfg.get('crop_medium_eff_thresh',120):
        win=int(eff*cfg.get('crop_medium_multiplier',1.8))
        win=max(cfg.get('crop_medium_min_px',80),min(win,cfg.get('crop_medium_max_px',260)))
    else:
        win=int(eff*cfg.get('crop_large_multiplier',1.2))
        win=max(cfg.get('crop_large_min_px',140),min(win,cfg.get('crop_large_max_px',320)))
    pr=cfg.get('crop_pad_ratio',0.15)
    x1=max(0,x-int(w*pr)); y1=max(0,y-int(h*pr))
    x2=min(Wi,x+w+int(w*pr)); y2=min(Hi,y+h+int(h*pr))
    return image[y1:y2,x1:x2]

def save_debug(name,image,dets,debug_dir,zone,cfg):
    if cfg.get('debug_outputs','annotate')=='none': return
    overlay=image.copy()
    for det in dets:
        x,y,w,h=det['bbox']
        color=(0,255,0) if is_near_flower((x,y,w,h),zone,cfg) else (255,0,0)
        cv2.rectangle(overlay,(x,y),(x+w,y+h),color,2)
        cv2.putText(overlay,det.get('candidate_type','')[:8],(x,y-6),
                    cv2.FONT_HERSHEY_SIMPLEX,0.38,color,1)
    q=int(cfg.get('debug_jpeg_quality',60)); mw=cfg.get('debug_max_width',1024)
    h_,w_=overlay.shape[:2]
    if mw and w_>mw:
        sc=mw/w_; overlay=cv2.resize(overlay,(mw,int(h_*sc)),cv2.INTER_AREA); scale=sc
    else: scale=1.0
    cv2.imwrite(str(Path(debug_dir)/f'{name}_4_final_saved_crops.jpg'),overlay,
                [cv2.IMWRITE_JPEG_QUALITY,q])
    (Path(debug_dir)/f'{name}_4_offset.json').write_text(
        json.dumps({'ox':0,'oy':0,'scale':scale}))

def get_exif_meta(path,cfg):
    m={'datetime':'','camera_name':'','shutter_speed':'','weather':'unknown',
       'laplacian_var':-1.0,'skip':False,'skip_reason':''}
    try:
        pil=PILImage.open(path)
        exif=pil._getexif() if hasattr(pil,'_getexif') else dict(pil.getexif())
        if exif:
            tags={TAGS.get(k,k):v for k,v in exif.items()}
            m['datetime']=str(tags.get('DateTimeOriginal',''))
            m['camera_name']=str(tags.get('Model',''))
            exp=exif.get(33434)
            if exp:
                d=exp[1] if isinstance(exp,tuple) else int(1/exp)
                m['shutter_speed']=f'1/{d}'
                m['weather']='sunny' if d>cfg['sunny_shutter_threshold'] else 'cloudy'
            fv=exif.get(37385,0)
            if fv and int(fv)!=0 and cfg.get('skip_flash',True):
                m['skip']=True; m['skip_reason']=f'flash(EXIF={fv})'
    except Exception: pass
    try:
        img=cv2.imread(str(path))
        if img is not None:
            lv=cv2.Laplacian(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),cv2.CV_64F).var()
            m['laplacian_var']=round(float(lv),1)
            if not m['skip'] and cfg.get('skip_foggy',True) and lv<cfg.get('foggy_threshold',50):
                m['skip']=True; m['skip_reason']=f'foggy(lap={lv:.1f})'
    except Exception: pass
    return m

def preprocess_image(img,cfg):
    sh=int(cfg.get('strip_height',0))
    if sh<=0 or img is None: return img,None
    h=img.shape[0]
    return (img[:h-sh,:],None) if sh<h else (img,None)

def is_near_flower(bbox,zone,cfg):
    x,y,w,h=bbox; roi=zone[y:y+h,x:x+w]
    return roi.size>0 and np.count_nonzero(roi)/roi.size>cfg['near_flower_iou_threshold']

def init_csv(path,fields):
    with open(path,'w',newline='') as f: csv.DictWriter(f,fieldnames=fields).writeheader()

def write_row(path,row,fields):
    with open(path,'a',newline='') as f:
        csv.DictWriter(f,fieldnames=fields).writerow({k:row.get(k,'') for k in fields})

def get_leaf_dirs(root):
    root=Path(root)
    if (any(root.glob('*.JPG')) or any(root.glob('*.jpg'))) and        not any(p.is_dir() for p in root.iterdir()): return [root]
    return [d for d in sorted(root.rglob('*'))
            if d.is_dir() and not any(x.is_dir() for x in d.iterdir())
            and (any(d.glob('*.JPG')) or any(d.glob('*.jpg')))]

print('✓ Preprocessing functions loaded.')


In [ ]:
import torch, torch.nn as nn, torchvision, torchvision.transforms as T

BINARY_CLASSES = ['background','insect']
GROUP_CLASSES  = ['bumblebee','fly','butterfly','other']
FIVE_CLASSES   = ['bumblebee','fly','butterfly','other','background']

def _letterbox(img,size):
    w,h=img.size; ms=max(w,h)
    sq=PILImage.new('RGB',(ms,ms),(0,0,0))
    sq.paste(img,((ms-w)//2,(ms-h)//2))
    return sq.resize((size,size),PILImage.BILINEAR)

def _load_single_model(path, default_classes, label=''):
    dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt=torch.load(path,map_location=dev,weights_only=False)
    cls=ckpt.get('classes',default_classes); sz=ckpt.get('img_size',224)
    keys=list(ckpt['state_dict'].keys())
    if any('trunk_output' in k or 'stem.' in k for k in keys):
        m=torchvision.models.regnet_y_32gf(weights=None); m.fc=nn.Linear(3712,len(cls))
    else:
        m=torchvision.models.efficientnet_b2(weights=None)
        m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,len(cls))
    m.load_state_dict(ckpt['state_dict']); m.eval()
    tf=T.Compose([T.Lambda(lambda i:_letterbox(i,sz)),T.ToTensor(),
                  T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    backbone='insectnet' if any('trunk_output' in k for k in keys) else 'efficientnet'
    print(f'  ✓ {label}: {backbone}  classes={cls}  img={sz}px')
    return m, tf, cls

def load_pipelines(pipelines_cfg):
    """
    Load all enabled pipelines.
    Returns dict: pipeline_name -> loaded bundle
    """
    loaded = {}
    for name, cfg in pipelines_cfg.items():
        if not cfg.get('enabled', True): continue
        print(f'  Loading pipeline: {name} ({cfg["type"]})')
        ptype = cfg['type']
        if ptype == 'two_stage':
            bm,btf,_ = _load_single_model(cfg['binary_model'], BINARY_CLASSES, 'binary')
            gm,gtf,gc= _load_single_model(cfg['group_model'],  GROUP_CLASSES,  'group')
            loaded[name] = {'type':'two_stage','bm':bm,'btf':btf,'gm':gm,'gtf':gtf,'gc':gc}
        elif ptype == 'five_class':
            fm,ftf,fc = _load_single_model(cfg['model'], FIVE_CLASSES, 'five_class')
            loaded[name] = {'type':'five_class','fm':fm,'ftf':ftf,'fc':fc}
        else:
            print(f'  ⚠ Unknown type: {ptype}')
    print(f'✓ {len(loaded)} pipeline(s) loaded.')
    return loaded

def _to_tensor_batch(crops_bgr, tf):
    imgs=[tf(PILImage.fromarray(cv2.cvtColor(c,cv2.COLOR_BGR2RGB))) for c in crops_bgr]
    return torch.stack(imgs)

def predict_all_pipelines(crops_bgr, loaded_pipelines, batch_size=32):
    """
    Run all loaded pipelines on a list of crops.
    Returns dict: pipeline_name -> list of result dicts (one per crop)
    """
    results = {name: [None]*len(crops_bgr) for name in loaded_pipelines}
    n = len(crops_bgr)
    if n == 0: return results

    with torch.no_grad():
        for start in range(0, n, batch_size):
            batch = crops_bgr[start:start+batch_size]
            idx   = list(range(start, start+len(batch)))

            for pipe_name, bundle in loaded_pipelines.items():
                ptype = bundle['type']

                if ptype == 'two_stage':
                    # Binary pass
                    xb = _to_tensor_batch(batch, bundle['btf'])
                    pb = torch.softmax(bundle['bm'](xb), 1)
                    bi = pb.argmax(1).tolist()
                    bc = pb.max(1).values.tolist()
                    # Group pass (only for insect predictions)
                    insect_idx = [i for i,b in enumerate(bi) if BINARY_CLASSES[b]=='insect']
                    group_res  = {}
                    if insect_idx:
                        xg = _to_tensor_batch([batch[i] for i in insect_idx], bundle['gtf'])
                        pg = torch.softmax(bundle['gm'](xg), 1)
                        for j,orig in enumerate(insect_idx):
                            gi=pg[j].argmax().item()
                            ap={c:float(pg[j][k]) for k,c in enumerate(bundle['gc'])}
                            group_res[orig]={
                                'pollinator_type': bundle['gc'][gi],
                                'group_conf':      round(float(pg[j][gi]),4),
                                'bumblebee_prob':  round(ap.get('bumblebee',0),4),
                                'fly_prob':        round(ap.get('fly',0),4),
                                'butterfly_prob':  round(ap.get('butterfly',0),4),
                                'other_prob':      round(ap.get('other',0),4),
                            }
                    for j,orig_i in enumerate(idx):
                        bl=BINARY_CLASSES[bi[j]]
                        r={'binary_label':bl,'binary_conf':round(bc[j],4),
                           'pollinator_type':'','group_conf':'',
                           'bumblebee_prob':'','fly_prob':'','butterfly_prob':'','other_prob':''}
                        if j in group_res: r.update(group_res[j])
                        results[pipe_name][orig_i] = r

                elif ptype == 'five_class':
                    xf = _to_tensor_batch(batch, bundle['ftf'])
                    pf = torch.softmax(bundle['fm'](xf), 1)
                    for j,orig_i in enumerate(idx):
                        fi=pf[j].argmax().item()
                        ap5={c:float(pf[j][k]) for k,c in enumerate(bundle['fc'])}
                        results[pipe_name][orig_i]={
                            'pollinator_type':  bundle['fc'][fi],
                            'conf':             round(float(pf[j][fi]),4),
                            'bumblebee_prob':   round(ap5.get('bumblebee',0),4),
                            'fly_prob':         round(ap5.get('fly',0),4),
                            'butterfly_prob':   round(ap5.get('butterfly',0),4),
                            'other_prob':       round(ap5.get('other',0),4),
                            'background_prob':  round(ap5.get('background',0),4),
                        }
    return results

def make_csv_fields(pipeline_names):
    """Generate CSV fields based on which pipelines are active."""
    base = [
        'camera_folder','image_name','datetime','temperature_c',
        'camera_name','shutter_speed','weather',
        'skip','skip_reason','laplacian_var','pollinator_detected',
        'crop_filename','bbox_x','bbox_y','bbox_w','bbox_h',
        'candidate_type','static_suspect','detection_scope','near_marked_flower',
    ]
    for name in pipeline_names:
        # We'll use pipeline name as prefix in CSV columns
        base += [f'{name}__binary_label', f'{name}__binary_conf',
                 f'{name}__pollinator_type', f'{name}__group_conf',
                 f'{name}__bumblebee_prob', f'{name}__fly_prob',
                 f'{name}__butterfly_prob', f'{name}__other_prob',
                 f'{name}__background_prob']
    return base

def pipeline_result_to_row(pipe_name, result):
    """Flatten a pipeline result dict into CSV row fields with prefix."""
    if result is None: return {}
    p = pipe_name + '__'
    ptype_fields = {}
    if 'binary_label' in result:  # two_stage
        ptype_fields = {
            p+'binary_label':   result.get('binary_label',''),
            p+'binary_conf':    result.get('binary_conf',''),
            p+'pollinator_type':result.get('pollinator_type',''),
            p+'group_conf':     result.get('group_conf',''),
            p+'bumblebee_prob': result.get('bumblebee_prob',''),
            p+'fly_prob':       result.get('fly_prob',''),
            p+'butterfly_prob': result.get('butterfly_prob',''),
            p+'other_prob':     result.get('other_prob',''),
            p+'background_prob':'',
        }
    else:  # five_class
        ptype_fields = {
            p+'binary_label':   '',
            p+'binary_conf':    '',
            p+'pollinator_type':result.get('pollinator_type',''),
            p+'group_conf':     result.get('conf',''),
            p+'bumblebee_prob': result.get('bumblebee_prob',''),
            p+'fly_prob':       result.get('fly_prob',''),
            p+'butterfly_prob': result.get('butterfly_prob',''),
            p+'other_prob':     result.get('other_prob',''),
            p+'background_prob':result.get('background_prob',''),
        }
    return ptype_fields

print('✓ Classifier + multi-pipeline functions loaded.')


In [ ]:
BATCH_SIZE = 32

def _save_run_config(run_dir, preprocess_cfg, pipelines_cfg):
    import json as _j
    def _safe(v):
        try: _j.dumps(v); return v
        except TypeError: return str(v)
    out = {
        'run_type':   'crop',
        'preprocess': {k:_safe(v) for k,v in preprocess_cfg.items()},
        'pipelines':  {name:{k:_safe(v) for k,v in cfg.items()}
                       for name,cfg in pipelines_cfg.items()
                       if cfg.get('enabled',True)},
    }
    (Path(run_dir)/'run_config.json').write_text(_j.dumps(out,indent=2))
    print(f'  ✓ run_config.json saved')

def get_leaf_dirs(root):
    root=Path(root)
    if (any(root.glob('*.JPG')) or any(root.glob('*.jpg'))) and        not any(p.is_dir() for p in root.iterdir()): return [root]
    return [d for d in sorted(root.rglob('*'))
            if d.is_dir() and not any(x.is_dir() for x in d.iterdir())
            and (any(d.glob('*.JPG')) or any(d.glob('*.jpg')))]

def run_folder(camera_dir, results_dir, preprocess_cfg, loaded_pipelines, csv_fields):
    camera_dir=Path(camera_dir); results_dir=Path(results_dir)
    crop_dir  =results_dir/'crops';  crop_dir.mkdir(parents=True,exist_ok=True)
    debug_dir =results_dir/'debug';  debug_dir.mkdir(exist_ok=True)
    csv_path  =results_dir/'results.csv'

    all_imgs=sorted(
        list(camera_dir.glob('*.JPG'))+list(camera_dir.glob('*.jpg')),
        key=robust_sort_key if preprocess_cfg.get('use_exif_sort') else filename_sort_key)
    if not all_imgs:
        print(f'  [SKIP] No images in {camera_dir.name}'); return 0, {}

    cam=camera_dir.name; total=len(all_imgs)
    print(f'  Images      : {total}')
    print(f'  large_motion: {"ON" if preprocess_cfg.get("enable_large_motion",True) else "OFF"}')
    print(f'  Pipelines   : {list(loaded_pipelines.keys()) if loaded_pipelines else "none (preprocess only)"}')

    zone,_ = setup_roi(all_imgs, preprocess_cfg)
    bg     = build_background(all_imgs, preprocess_cfg)
    init_csv(csv_path, csv_fields)

    win=preprocess_cfg.get('rolling_window',0); cache=[]
    n_bbox=0; n_skipped=0; n_no_det=0

    # Per-pipeline counters
    pipe_counts = {name: {
        'insect':0,'background':0,
        'bumblebee':0,'fly':0,'butterfly':0,'other':0,
    } for name in (loaded_pipelines or {})}

    # Batch accumulation
    pending_crops = []
    pending_meta  = []

    def flush_batch():
        if not pending_crops: return
        if loaded_pipelines:
            pipe_results = predict_all_pipelines(pending_crops, loaded_pipelines, BATCH_SIZE)
        else:
            pipe_results = {}

        for i,(row_base, fname, crop) in enumerate(pending_meta):
            cv2.imwrite(str(crop_dir/fname), crop)
            full_row = dict(row_base)
            for pipe_name in (loaded_pipelines or {}):
                res = pipe_results.get(pipe_name,[None]*len(pending_crops))[i]
                full_row.update(pipeline_result_to_row(pipe_name, res))
                # Count insect vs background per pipeline
                if res:
                    if 'binary_label' in res:  # two_stage
                        k = 'insect' if res.get('binary_label')=='insect' else 'background'
                    else:  # five_class
                        pt = res.get('pollinator_type','')
                        k = 'background' if pt=='background' or not pt else 'insect'
                    pipe_counts[pipe_name][k] += 1
                    # Count pollinator type
                    if res:
                        pt = res.get('pollinator_type','')
                        if pt and pt not in ('background',''):
                            pipe_counts[pipe_name][pt] = \
                                pipe_counts[pipe_name].get(pt,0) + 1
            write_row(csv_path, full_row, csv_fields)
        pending_crops.clear(); pending_meta.clear()

    for idx, path in enumerate(all_imgs):
        img=cv2.imread(str(path))
        if img is None: continue
        img,_ = preprocess_image(img, preprocess_cfg)
        if zone.shape[0]!=img.shape[0]: zone=zone[:img.shape[0],:img.shape[1]]

        bg_use=(cache[-1] if win==1 else np.median(cache[-win:],axis=0).astype(np.uint8))                if win>0 and cache else bg
        dets=[]
        if bg_use is not None:
            bgu=bg_use[:img.shape[0],:img.shape[1]] if bg_use.shape[:2]!=img.shape[:2] else bg_use
            dets=detect_visitor(img,bgu,zone,preprocess_cfg)
        cache.append(img)
        if win>0 and len(cache)>win: cache.pop(0)

        meta=get_exif_meta(path,preprocess_cfg); meta['camera_folder']=cam

        if preprocess_cfg.get('debug_outputs','annotate')!='none' and dets:
            save_debug(f'{cam}__{path.stem}',img,dets,debug_dir,zone,preprocess_cfg)

        if meta.get('skip'):
            n_skipped+=1; flush_batch()
            try: rel_path = str(path.relative_to(camera_dir.parent.parent))
            except ValueError: rel_path = str(path)
            write_row(csv_path,{'image_name':path.name,'image_path':rel_path,**meta,
                'pollinator_detected':'skipped'},csv_fields)
            print(f'  [{idx+1:>4}/{total}] {path.name}  SKIP: {meta["skip_reason"]}', flush=True)
            continue

        if not dets:
            n_no_det+=1
            try: rel_path = str(path.relative_to(camera_dir.parent.parent))
            except ValueError: rel_path = str(path)
            write_row(csv_path,{'image_name':path.name,'image_path':rel_path,**meta,
                'pollinator_detected':'no'},csv_fields)
            continue

        # Has detections — print and queue for batch inference
        n_bbox += len(dets)
        print(f'  [{idx+1:>4}/{total}] {path.name}  {len(dets)} bbox', flush=True)

        for i,det in enumerate(dets):
            x,y,w,h=det['bbox']
            ct=det.get('candidate_type','normal')
            ss=det.get('static_suspect',False)
            in_r=is_near_flower((x,y,w,h),zone,preprocess_cfg)
            scope='roi' if in_r else 'out'
            crop=crop_with_padding(img,det,preprocess_cfg)
            if crop is None or crop.size==0: continue
            fname=f'{cam}__{path.stem}_crop{i:02d}_{ct[:6]}_{scope}.jpg'
            # Store relative path from IMAGE_ROOT for cross-platform compatibility
            try: rel_path = str(path.relative_to(camera_dir.parent.parent))
            except ValueError: rel_path = str(path)
            row_base={
                'image_name':path.name,
                'image_path':rel_path,
                **meta,
                'pollinator_detected':'yes',
                'crop_filename':fname,
                'bbox_x':x,'bbox_y':y,'bbox_w':w,'bbox_h':h,
                'candidate_type':ct,'static_suspect':str(ss),
                'near_marked_flower':str(in_r),
                'detection_scope':'roi' if in_r else 'outside_roi',
            }
            pending_crops.append(crop)
            pending_meta.append((row_base, fname, crop))
            if len(pending_crops)>=BATCH_SIZE: flush_batch()

    flush_batch()

    # ── Per-folder summary ────────────────────────────────────────
    print(f'  ─────────────────────────────────────────')
    print(f'  {total} frames  |  {n_bbox} bbox detected  |  skip={n_skipped}')
    if loaded_pipelines:
        POLL_CLASSES = ['bumblebee','fly','butterfly','other']
        for pipe_name, counts in pipe_counts.items():
            ins = counts['insect']; bg = counts['background']
            cls_str = '  '.join(f'{c}={counts.get(c,0)}' for c in POLL_CLASSES)
            print(f'  {pipe_name:20}: insect={ins:<5} background={bg:<5}  [{cls_str}]')

    return n_bbox, pipe_counts

print('✓ Pipeline functions loaded.')


## Cell 3 — Run config  ← **edit this before every run**

Three things to set:

1. **`RUN_NAME`** — a short descriptive name for this experiment.
   Results go into `crop_results/{RUN_NAME}/`. Change this each time so runs don't overwrite each other.
   Examples: `'run_lm_on'`, `'run_thr25'`, `'run_01'`

2. **`PREPROCESS_CONFIG`** — detection parameters.
   Key toggles to experiment with:
   - `enable_large_motion`: True/False — include large-motion region tiling
   - `darker_threshold`: lower = more sensitive detection, more false positives
   - `kernel_close_size`: larger = merges more fragments into one detection

3. **`PIPELINES`** — which classifier models to run.
   Add/remove entries freely. Set `enabled: False` to skip a pipeline without deleting it.
   Each enabled pipeline adds 9 columns to the CSV (prefixed by pipeline name).

In [ ]:
# ── Run name ─────────────────────────────────────────────────────
# Change this for each experiment so results don't overwrite each other
# e.g. 'run_lm_on', 'run_lm_off', 'run_thr25', 'run_preprocess_only'
RUN_NAME = 'run_01'

# ── Preprocessing config ─────────────────────────────────────────
# Only change what you want to vary from defaults
PREPROCESS_CONFIG = {
    **DEFAULT_PREPROCESS_CONFIG,
    'enable_large_motion': True,    # ← key toggle
    'darker_threshold':    30,      # ← key toggle
    'rolling_window':      1,
    'use_roi':             False,
}

# ── Pipelines to run ──────────────────────────────────────────────
# Add / remove / disable any pipeline here
# Each enabled pipeline adds columns to the CSV: {name}__binary_label, etc.
PIPELINES = {
    'two_stage': {
        'enabled':      True,
        'type':         'two_stage',
        'binary_model': MODEL_DIR / 'binary_best.pth',
        'group_model':  MODEL_DIR / '4group_insectnet.pth',
    },
    'five_class_eff': {
        'enabled':      True,
        'type':         'five_class',
        'model':        MODEL_DIR / '5group_efficientnet.pth',
    },
    'five_class_ins': {
        'enabled':      True,
        'type':         'five_class',
        'model':        MODEL_DIR / '5group_insectnet.pth',
    },
    # ── Add more pipelines here as needed ──────────────────────
    # 'two_stage_v2': {
    #     'enabled':      False,
    #     'type':         'two_stage',
    #     'binary_model': MODEL_DIR / 'binary_v2.pth',
    #     'group_model':  MODEL_DIR / '4group_v2.pth',
    # },
}

RUN_DIR = CROP_RESULTS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'RUN_NAME  : {RUN_NAME}')
print(f'Results → : {RUN_DIR}')
active = [n for n,c in PIPELINES.items() if c.get('enabled',True)]
print(f'Pipelines : {active}')
print(f'large_motion: {PREPROCESS_CONFIG["enable_large_motion"]}')
print(f'darker_threshold: {PREPROCESS_CONFIG["darker_threshold"]}')


## Cell 4 — Verify model files

Checks that all model `.pth` files exist before loading them.
If any file shows `✗ NOT FOUND`, fix the path in Cell 2 before continuing.

In [ ]:
print('Verifying model files...')
all_ok = True
for pipe_name, cfg in PIPELINES.items():
    if not cfg.get('enabled', True): continue
    paths_to_check = []
    if cfg['type'] == 'two_stage':
        paths_to_check = [cfg['binary_model'], cfg['group_model']]
    else:
        paths_to_check = [cfg['model']]
    for p in paths_to_check:
        ok = Path(p).exists()
        print(f'  [{pipe_name}] {Path(p).name}: {"✓" if ok else "✗ NOT FOUND"}')
        if not ok: all_ok = False
assert all_ok, 'Some model files missing — check paths in Cell 2'


## Cell 5 — Load models + find camera folders

- Loads all enabled pipeline models into memory
- Scans `IMAGE_ROOT` for camera folders (leaf directories containing `.JPG` files)
- Shows a count of images per folder so you know what to expect

If a camera folder shows 0 images, check that `IMAGE_ROOT` is correct.

In [ ]:
loaded_pipelines = load_pipelines(PIPELINES)
csv_fields = make_csv_fields(list(loaded_pipelines.keys()))
print(f'\nCSV will have {len(csv_fields)} columns')

assert IMAGE_ROOT.exists(), f'IMAGE_ROOT not found: {IMAGE_ROOT}'
camera_dirs = get_leaf_dirs(IMAGE_ROOT)
total_images = 0
print(f'\nFound {len(camera_dirs)} camera folder(s):')
for d in camera_dirs:
    n = len(list(d.glob('*.JPG'))) + len(list(d.glob('*.jpg')))
    total_images += n
    print(f'  {d.name:<55} {n:>5} images')
print(f'  {"─"*62}')
print(f'  Total: {total_images} images')


## Cell 6 — Run  ← **this is the main processing cell**

Processes all camera folders one by one:
1. Saves `run_config.json` for traceability
2. For each camera folder:
   - Detects candidate insects frame-by-frame (rolling window background subtraction)
   - Saves all candidate crops to `crops/`
   - Runs batch inference through all enabled pipelines
   - Writes one CSV row per crop with results from all pipelines
   - Saves annotated debug frames to `debug/`

**Progress** is printed every 50 frames showing fps and ETA.
Press `■ Stop` to interrupt safely — partial results are already saved.

In [ ]:
import traceback

_save_run_config(RUN_DIR, PREPROCESS_CONFIG, PIPELINES)

# Accumulators for final summary
total_stats = {'frames': 0, 'bbox': 0}
pipeline_totals = {name: {
        'insect':0,'background':0,
        'bumblebee':0,'fly':0,'butterfly':0,'other':0,
    } for name in loaded_pipelines}

t_start = time.time()

for ci, camera_dir in enumerate(camera_dirs):
    out = RUN_DIR / camera_dir.name
    out.mkdir(exist_ok=True)
    print(f'\n[{ci+1}/{len(camera_dirs)}] {camera_dir.name}')
    print(f'  ─────────────────────────────────────────')
    try:
        n_bbox, pipe_counts = run_folder(
            camera_dir, out, PREPROCESS_CONFIG, loaded_pipelines, csv_fields)
        total_stats['frames'] += len(list(camera_dir.glob('*.JPG')))+len(list(camera_dir.glob('*.jpg')))
        total_stats['bbox']   += n_bbox
        for pipe_name, counts in pipe_counts.items():
            for k in ('insect','background','bumblebee','fly','butterfly','other'):
                pipeline_totals[pipe_name][k] += counts.get(k,0)
    except KeyboardInterrupt:
        print('\n⚠ Interrupted.'); raise
    except Exception as e:
        print(f'  ✗ ERROR: {e}'); traceback.print_exc()

total_time = time.time() - t_start

# ── Final summary ─────────────────────────────────────────────────
print(f'\n{"═"*55}')
print(f'RUN COMPLETE : {RUN_NAME}')
print(f'  Total frames : {total_stats["frames"]}')
print(f'  Total bbox   : {total_stats["bbox"]}')
print(f'  Time         : {total_time:.1f}s ({total_time/60:.1f} min)')
print()
POLL_CLASSES = ['bumblebee','fly','butterfly','other']
if pipeline_totals:
    pipe_names = list(pipeline_totals.keys())
    col_w = max(len(n) for n in pipe_names) + 2
    # Detection table
    print(f'  {"Pipeline":{col_w}}  {"insect":>8}  {"background":>10}  {"total":>7}')
    print(f'  {"-"*(col_w+30)}')
    for pipe_name, counts in pipeline_totals.items():
        ins = counts['insect']; bg = counts['background']
        print(f'  {pipe_name:{col_w}}  {ins:>8}  {bg:>10}  {ins+bg:>7}')
    # Pollinator breakdown table
    print()
    print(f'  Pollinator breakdown (insect crops only):')
    header = f'  {"Pipeline":{col_w}}  ' + '  '.join(f'{c:>10}' for c in POLL_CLASSES)
    print(header)
    print(f'  {"-"*(col_w+50)}')
    for pipe_name, counts in pipeline_totals.items():
        row = f'  {pipe_name:{col_w}}  ' + '  '.join(f'{counts.get(c,0):>10}' for c in POLL_CLASSES)
        print(row)
print(f'{"═"*55}')
print(f'Results : {RUN_DIR}')

# ── Auto-save results to Drive (Colab only) ──────────────────────
if IN_COLAB:
    import shutil
    drive_results = Path('/content/drive/MyDrive/pollinator-colab/Insects_images/crop_results')
    drive_results.mkdir(parents=True, exist_ok=True)
    drive_run_dir = drive_results / RUN_NAME
    if drive_run_dir.exists():
        shutil.rmtree(str(drive_run_dir))
    print(f'\nSaving results to Drive...')
    shutil.copytree(str(RUN_DIR), str(drive_run_dir))
    print(f'✓ Results saved to Drive: {drive_run_dir}')
    print(f'  evaluate.ipynb can read from Drive even in a new session.')


  ✓ run_config.json saved

[1/7] e2e_hdd1_salix_vamy_p2
  ─────────────────────────────────────────
  Images      : 175
  large_motion: ON
  Pipelines   : ['two_stage', 'five_class_eff', 'five_class_ins']
  ROI: disabled — full image
  Background: rolling window only
  [   4/175] WSCT0910.JPG  18 bbox
  [   5/175] WSCT0911.JPG  16 bbox
  [   8/175] WSCT1085.JPG  125 bbox
  [  10/175] WSCT1087.JPG  4 bbox
  [  11/175] WSCT1088.JPG  1 bbox
  [  12/175] WSCT1089.JPG  9 bbox
  [  13/175] WSCT1090.JPG  11 bbox
  [  14/175] WSCT1091.JPG  1 bbox
  [  15/175] WSCT1179.JPG  51 bbox
  [  16/175] WSCT1180.JPG  7 bbox
  [  17/175] WSCT1181.JPG  11 bbox
  [  18/175] WSCT1182.JPG  19 bbox
  [  19/175] WSCT1183.JPG  7 bbox
  [  20/175] WSCT1199.JPG  143 bbox
  [  21/175] WSCT1200.JPG  2 bbox
  [  22/175] WSCT1201.JPG  20 bbox
  [  23/175] WSCT1202.JPG  18 bbox


## Cell 7 — Quick summary

After the run completes, shows a breakdown per pipeline:
how many crops were classified as insect vs background.

Useful for a quick sanity check before running `evaluate.ipynb`.
If all pipelines show 0 insects, something went wrong with detection or models.

In [ ]:
all_csvs = list(RUN_DIR.rglob('results.csv'))
rows_yes = []
for f in all_csvs:
    with open(f,newline='') as fh:
        rows_yes += [r for r in csv.DictReader(fh) if r.get('pollinator_detected')=='yes']

print(f'Run      : {RUN_NAME}')
print(f'Crops    : {len(rows_yes)} candidates')
print()
for pipe_name, bundle in loaded_pipelines.items():
    p = pipe_name + '__'
    if bundle['type'] == 'two_stage':
        ins = sum(1 for r in rows_yes if r.get(p+'binary_label')=='insect')
        bg  = sum(1 for r in rows_yes if r.get(p+'binary_label')=='background')
        print(f'  {pipe_name:20} → insect={ins}  background={bg}')
    else:
        ins = sum(1 for r in rows_yes if r.get(p+'pollinator_type') not in ('background',''))
        bg  = sum(1 for r in rows_yes if r.get(p+'pollinator_type')=='background')
        print(f'  {pipe_name:20} → insect={ins}  background={bg}')
